# ⚗️ IRMS Analysis: Geographic Origin of Olive Oil
### FBMFOR — Food Fraud Analysis | University of Reading

This notebook guides you through:
1. Loading stable isotope data from `irms.csv`
2. Visualising **δ²H vs δ¹⁸O** — the primary geolocation plot
3. Visualising **δ¹³C vs δ¹⁵N** — agricultural practice profile (if data available)
4. Overlaying **literature reference regions** from Torres-Cobos et al. (2025)
5. Assigning **unknown samples** to a probable geographic origin

---
> **Fraud context:** Extra virgin olive oil commands a significant price premium based on geographic origin (PDO status).  
> Italian EVOO trades at ~35% above other European oils. Stable isotope ratios cannot be manipulated and act  
> as natural geographic fingerprints — making IRMS one of the most powerful tools for origin verification.

## 1 · Setup

In [ ]:
# ============================================================
# SETUP — INSTALL AND IMPORT LIBRARIES
# ============================================================
# All libraries used here are pre-installed in Google Colab.
# If running locally, install with:  pip install numpy pandas matplotlib scipy

# --- Core numerical and data libraries ---
import numpy as np                        # Array operations and numerical computation
import pandas as pd                       # DataFrames for tabular data handling

# --- Plotting ---
import matplotlib.pyplot as plt           # Static publication-quality figures
import matplotlib.patches as mpatches    # Custom legend patches
from matplotlib.patches import Ellipse   # For drawing reference ellipses
import matplotlib.patheffects as pe      # Text outline effects

# --- Statistics ---
from scipy import stats                   # For confidence ellipses

import os, io, warnings
warnings.filterwarnings('ignore')

# --- Colab file upload helper ---
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("\u2713 Libraries loaded")
print(f"  Running in Google Colab: {IN_COLAB}")

## 2 · Literature Reference Data

Reference ranges compiled from the published literature.  
The primary source is Torres-Cobos et al. (2025), a study of ~400 virgin olive oil samples  
across four harvest seasons (2016–2020) from six Mediterranean countries.  
Additional regional ranges are drawn from the review by Longobardi et al. (2012).

Edit this cell to add or refine regions as new reference data become available.

In [ ]:
# ============================================================
# LITERATURE REFERENCE REGIONS
# ============================================================
# Each entry defines the approximate centre and spread of the
# isotope signature for a given country/region, derived from
# published studies.
#
# δ¹⁸O is the most powerful single discriminator for Italian vs
# non-Italian oils (p < 0.001; Torres-Cobos et al., 2025).
# Italian oils are systematically LOWER in δ¹⁸O than other
# Mediterranean oils — likely reflecting lower temperatures and
# higher precipitation in northern Italian growing regions.
#
# δ²H and δ¹⁸O co-vary along the Global Meteoric Water Line
# (δ²H ≈ 8 × δ¹⁸O + 10), but evapotranspiration causes
# enrichment in heavy isotopes beyond this baseline.
#
# Values are medians with approximate 1 SD spread from Torres-Cobos
# et al. (2025) and supplementary regional data.
#
# Sources:
#   Torres-Cobos et al. (2025) Food Chem. doi:10.1016/j.foodchem.2024.141998
#   Longobardi et al. (2012) Food Chem. 131, 1149–1154.
#   Bontempo et al. (2019) Food Control 98, 482–489.

# --- δ²H / δ¹⁸O reference regions ---
# Format: { 'Country': { 'd18O_mean', 'd18O_sd', 'd2H_mean', 'd2H_sd', 'colour' } }

REF_HO = {
    "Italy": {
        "d18O_mean": 24.0,  "d18O_sd": 1.8,
        "d2H_mean":  -144.0, "d2H_sd":  12.0,
        # Italian oils show lower δ¹⁸O than other Mediterranean oils
        # (Torres-Cobos et al. 2025, median 24.0 vs 25.8‰ non-Italian)
        "colour": "#2E86AB",   # Blue
        "n": 242
    },
    "Spain": {
        "d18O_mean": 26.0,  "d18O_sd": 2.0,
        "d2H_mean":  -140.0, "d2H_sd":  14.0,
        # Spain (mainly Andalusia): warmer, drier → more positive δ¹⁸O
        "colour": "#E84855",   # Red
        "n": 80
    },
    "Greece": {
        "d18O_mean": 26.5,  "d18O_sd": 2.2,
        "d2H_mean":  -136.0, "d2H_sd":  15.0,
        # Greek oils (islands + mainland): high δ¹⁸O due to Mediterranean climate
        "colour": "#3BB273",   # Green
        "n": 45
    },
    "Tunisia": {
        "d18O_mean": 27.5,  "d18O_sd": 1.8,
        "d2H_mean":  -128.0, "d2H_sd":  12.0,
        # North African oils: highest δ¹⁸O, least negative δ²H
        "colour": "#F4A261",   # Orange
        "n": 30
    },
}

# --- δ¹³C / δ¹⁵N reference values ---
# δ¹³C is less useful for country-level discrimination within
# Mediterranean olive oils (no significant difference between Italy
# and non-Italy in Torres-Cobos et al. 2025), but distinguishes:
#   • C3 plants (olives, -32 to -22‰) from
#   • C4 adulterants (sunflower, corn oil ≈ -16 to -9‰)
# δ¹⁵N distinguishes organic (manure) from synthetic fertilisation.

REF_CN = {
    "Olive oil (authentic)": {
        "d13C_mean": -29.6,  "d13C_sd": 1.2,
        "d15N_mean":  5.5,   "d15N_sd": 3.5,
        "colour": "#2E86AB",
    },
    "Sunflower oil (adulterant)": {
        "d13C_mean": -29.0,  "d13C_sd": 1.0,
        "d15N_mean":  4.0,   "d15N_sd": 3.0,
        # Also C3 — δ¹³C alone cannot distinguish from olive
        "colour": "#E9C46A",
    },
    "Corn oil (C4 adulterant)": {
        "d13C_mean": -14.0,  "d13C_sd": 1.5,
        "d15N_mean":  3.0,   "d15N_sd": 3.5,
        # C4 pathway — strongly enriched in ¹³C: clear fraud signature
        "colour": "#E76F51",
    },
}

print("\u2713 Reference data loaded")
print("\n  H/O geolocation regions:")
for region, vals in REF_HO.items():
    print(f"    {region:<12s}  δ¹⁸O = {vals['d18O_mean']:.1f} ± {vals['d18O_sd']:.1f}‰   "
          f"δ²H = {vals['d2H_mean']:.1f} ± {vals['d2H_sd']:.1f}‰  (n={vals['n']})")
print("\n  C/N agricultural practice regions:")
for region, vals in REF_CN.items():
    print(f"    {region:<30s}  δ¹³C = {vals['d13C_mean']:.1f} ± {vals['d13C_sd']:.1f}‰   "
          f"δ¹⁵N = {vals['d15N_mean']:.1f} ± {vals['d15N_sd']:.1f}‰")

## 3 · Load Data

The expected file is `irms.csv` with columns:

| Column | Units | Required? | Notes |
|--------|-------|-----------|-------|
| `SampleID` | — | ✓ | Sample label (used for point annotations) |
| `d2H` | ‰ vs VSMOW | ✓ | δ²H (deuterium/hydrogen ratio) |
| `d18O` | ‰ vs VSMOW | ✓ | δ¹⁸O (oxygen-18 ratio) |
| `d13C` | ‰ vs VPDB | optional | δ¹³C (carbon-13 ratio) |
| `d15N` | ‰ vs AIR | optional | δ¹⁵N (nitrogen-15 ratio) |

Loading is attempted in order: **GitHub → Colab upload → local file**.

In [ ]:
# ============================================================
# LOAD IRMS DATA
# ============================================================
# irms.csv must contain: SampleID, d2H, d18O
# Optional columns: d13C, d15N
#
# Columns d13C and d15N are read if present; their absence simply
# disables the C/N plot in Section 6.

GITHUB_URL = (
    "https://raw.githubusercontent.com/ggkuhnle/fbmfor-foodfraud/main/data/irms.csv"
)

df = None  # Will hold the loaded DataFrame

# --- 1: Try GitHub URL ---
try:
    df = pd.read_csv(GITHUB_URL)
    print(f"\u2713 Loaded data from GitHub ({len(df)} rows)")
except Exception as e:
    print(f"  GitHub load failed ({e}); trying upload/local …")

# --- 2: Colab file upload ---
if df is None and IN_COLAB:
    print("\U0001f4c2 Upload irms.csv:")
    uploaded = colab_files.upload()
    fname = list(uploaded.keys())[0]
    df = pd.read_csv(io.BytesIO(uploaded[fname]))
    print(f"\u2713 Loaded from upload ({len(df)} rows)")

# --- 3: Local file fallback ---
if df is None:
    for p in ["data/irms.csv", "irms.csv"]:
        if os.path.exists(p):
            df = pd.read_csv(p)
            print(f"\u2713 Loaded from {p} ({len(df)} rows)")
            break

if df is None:
    raise FileNotFoundError(
        "Could not load irms.csv. Check your internet connection or upload the file."
    )

# --- Validate ---
df.columns = [c.strip() for c in df.columns]  # Remove accidental whitespace
REQUIRED = {"SampleID", "d2H", "d18O"}
if not REQUIRED.issubset(df.columns):
    raise ValueError(
        f"Missing required columns. Found: {list(df.columns)}, need: {REQUIRED}"
    )

# --- Detect optional columns ---
HAS_C   = "d13C" in df.columns
HAS_N   = "d15N" in df.columns
HAS_CN  = HAS_C and HAS_N

# --- Report ---
print(f"\n  Columns present: {list(df.columns)}")
print(f"  δ¹³C available: {HAS_C}")
print(f"  δ¹⁵N available: {HAS_N}")
print(f"\n  Preview:")
df

## 4 · Summary Statistics

Calculate descriptive statistics for each isotope ratio across all samples.  
This is also the step where you would identify outliers before plotting.

In [ ]:
# ============================================================
# SUMMARY STATISTICS
# ============================================================
# Report mean, SD, min, max for each available isotope column.
# Flag any samples that deviate by more than OUTLIER_SD standard
# deviations from the column mean — these are worth scrutinising
# before drawing conclusions.

# ── PARAMETER ────────────────────────────────────────────────
OUTLIER_SD = 3.0   # Flag samples beyond this many SDs from the mean
# ─────────────────────────────────────────────────────────────

# Identify the numeric isotope columns that are present
iso_cols = [c for c in ["d2H", "d18O", "d13C", "d15N"] if c in df.columns]

# Pretty labels for display
col_labels = {
    "d2H":  "δ²H (‰ VSMOW)",
    "d18O": "δ¹⁸O (‰ VSMOW)",
    "d13C": "δ¹³C (‰ VPDB)",
    "d15N": "δ¹⁵N (‰ AIR)",
}

print("Summary statistics for all samples\n")
print(f"  {'Isotope':<22}  {'Mean':>8}  {'SD':>8}  {'Min':>8}  {'Max':>8}  {'N':>5}")
print("  " + "-" * 64)
for col in iso_cols:
    vals = df[col].dropna()
    print(f"  {col_labels[col]:<22}  "
          f"{vals.mean():8.2f}  {vals.std():8.2f}  "
          f"{vals.min():8.2f}  {vals.max():8.2f}  {len(vals):5d}")

# --- Outlier check ---
print("\nOutlier check:")
any_outliers = False
for col in iso_cols:
    vals = df[col].dropna()
    mu, sd = vals.mean(), vals.std()
    outliers = df[np.abs(df[col] - mu) > OUTLIER_SD * sd]
    if not outliers.empty:
        any_outliers = True
        for _, row in outliers.iterrows():
            print(f"  ⚠  {row['SampleID']}  {col_labels[col]} = {row[col]:.2f}  "
                  f"({(row[col]-mu)/sd:+.1f} SD from mean)")
if not any_outliers:
    print(f"  \u2713 No values beyond ±{OUTLIER_SD} SD — no flagged outliers.")

## 5 · δ²H vs δ¹⁸O — Geographic Origin Plot

This is the **primary geolocation plot**.  

Both isotopes are controlled by the local water cycle (precipitation, evapotranspiration),  
making their combination a powerful fingerprint for geographic origin.  
The shaded ellipses show 1-SD reference regions from Torres-Cobos et al. (2025).  

**Key principle:** Italian oils are systematically lower in δ¹⁸O than other Mediterranean oils  
(median 24.0‰ vs 25.8‰; p < 0.001), reflecting cooler, wetter growing conditions.

In [ ]:
# ============================================================
# δ²H vs δ¹⁸O PLOT  — GEOGRAPHIC ORIGIN
# ============================================================
# The ellipses represent ±1 SD around the published regional median
# for each axis independently. They are drawn as guides to the eye,
# not as formal decision boundaries — full classification would use
# LDA or QDA on a validated reference database.

# ── PARAMETERS — modify and re-run ────────────────────────────
ELLIPSE_SD    = 1.0   # SD multiplier for reference ellipses (1 = 1 SD, 2 = 2 SD)
LABEL_OFFSET  = (0.15, 1.5)  # (x, y) offset for sample ID labels in data units
SAVE_FIGURE   = True
FIGURE_DPI    = 200
FIGURE_NAME   = "irms_HO_origin.png"
# ─────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_facecolor("#FAFAFA")

# --- Reference ellipses ---
# Each ellipse is centred on the published median with width = 2*SD_x
# and height = 2*SD_y, scaled by ELLIPSE_SD.
for region, vals in REF_HO.items():
    cx  = vals["d18O_mean"]
    cy  = vals["d2H_mean"]
    w   = 2 * ELLIPSE_SD * vals["d18O_sd"]
    h   = 2 * ELLIPSE_SD * vals["d2H_sd"]
    col = vals["colour"]

    # Filled translucent ellipse
    ell = Ellipse((cx, cy), width=w, height=h,
                  facecolor=col, alpha=0.18, edgecolor=col,
                  linewidth=1.5, linestyle="--", zorder=1)
    ax.add_patch(ell)

    # Region label at the ellipse centre
    ax.text(cx, cy, region,
            fontsize=9, color=col, fontweight="bold", ha="center", va="center",
            zorder=3,
            path_effects=[pe.withStroke(linewidth=2, foreground="white")])

# --- Sample data points ---
for _, row in df.iterrows():
    sid = str(row["SampleID"])
    x   = row["d18O"]
    y   = row["d2H"]

    # Use a distinct symbol for samples labelled 'Unknown' (case-insensitive)
    is_unknown = "unknown" in sid.lower() or "unk" in sid.lower()
    marker = "D" if is_unknown else "o"   # Diamond for unknowns, circle for labelled
    colour = "#6C3483" if is_unknown else "#2C3E50"
    size   = 80        if is_unknown else 60

    ax.scatter(x, y, marker=marker, s=size, color=colour,
               edgecolors="white", linewidths=0.8, zorder=5)

    ax.annotate(sid,
                xy=(x, y),
                xytext=(x + LABEL_OFFSET[0], y + LABEL_OFFSET[1]),
                fontsize=8, color=colour, fontweight="bold",
                path_effects=[pe.withStroke(linewidth=2, foreground="white")],
                zorder=6)

# --- Axes and labels ---
ax.set_xlabel(r"$\delta^{18}$O (‰ vs VSMOW)", fontsize=12)
ax.set_ylabel(r"$\delta^{2}$H (‰ vs VSMOW)", fontsize=12)
ax.set_title(r"Stable Isotope Map: $\delta^{2}$H vs $\delta^{18}$O"
             "\nGeographic Origin of Virgin Olive Oil",
             fontsize=13, fontweight="bold", pad=10)

# --- Legend ---
legend_handles = [
    mpatches.Patch(facecolor=vals["colour"], alpha=0.5, edgecolor=vals["colour"],
                   linestyle="--", label=f"{region} (n={vals['n']})")
    for region, vals in REF_HO.items()
]
legend_handles += [
    plt.scatter([], [], marker="o", s=60, color="#2C3E50",
                edgecolors="white", linewidths=0.8, label="Sample (labelled)"),
    plt.scatter([], [], marker="D", s=80, color="#6C3483",
                edgecolors="white", linewidths=0.8, label="Sample (unknown)"),
]
ax.legend(handles=legend_handles,
          title=f"Reference regions (±{ELLIPSE_SD} SD)",
          title_fontsize=8.5, fontsize=8.5, loc="upper left", framealpha=0.9)

ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, linestyle=":", alpha=0.4)

ax.text(0.5, -0.10,
        "Ellipses: ±1 SD reference regions (Torres-Cobos et al., 2025; Longobardi et al., 2012). "
        "Diamonds = unknowns.",
        transform=ax.transAxes, ha="center", fontsize=8, color="grey")

plt.tight_layout()
if SAVE_FIGURE:
    plt.savefig(FIGURE_NAME, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"\u2713 Figure saved as {FIGURE_NAME}")
plt.show()

### How to interpret the δ²H / δ¹⁸O plot

**Why δ¹⁸O is the key discriminator for Italian oils:**  
Italian oils cluster at lower δ¹⁸O values (~24‰) compared to Spanish (~26‰), Greek (~26.5‰),  
and North African (~27.5‰) oils. This reflects the Rayleigh distillation effect: as moist air  
masses move inland over the Alps, heavy isotopes precipitate preferentially, leaving  
precipitation — and ultimately olive tree tissue — depleted in ¹⁸O.

**Why δ²H and δ¹⁸O co-vary:**  
Both isotopes are fractionated by the same evaporation/condensation processes.  
They follow the Global Meteoric Water Line (δ²H ≈ 8 × δ¹⁸O + 10), but evapotranspiration  
in olive leaves enriches plant water above this line — creating region-specific offsets.

**Limitations:**  
- Year-to-year climate variation can shift values (the 2018/19 harvest had higher misclassification)
- Irrigation shifts δ¹⁸O toward the isotopic composition of the irrigation water source
- Sub-regional discrimination (e.g., Sicily vs Apulia) requires multi-year reference databases

## 6 · δ¹³C vs δ¹⁵N — Agricultural Practice Profile

This plot is generated **only if both `d13C` and `d15N` columns are present** in `irms.csv`.  
If only one column is present, a 1D strip plot is shown instead.

**δ¹³C** distinguishes C3 plants (olives, sunflower, rapeseed: ~−32 to −22‰) from  
C4 adulterants (corn oil: ~−16 to −9‰). It cannot alone discriminate among C3 oils.

**δ¹⁵N** reflects the nitrogen source:  
- Synthetic fertiliser: δ¹⁵N ≈ 0‰  
- Organic manure: δ¹⁵N = +5 to +15‰  
- Legume-fixed N₂: δ¹⁵N ≈ 0‰  

Combined, δ¹³C / δ¹⁵N can detect C4 adulteration and indicate production practices.

In [ ]:
# ============================================================
# δ¹³C vs δ¹⁵N PLOT — AGRICULTURAL PRACTICE PROFILE
# ============================================================

# ── PARAMETERS ───────────────────────────────────────────────
SAVE_FIGURE_CN = True
FIGURE_NAME_CN = "irms_CN_practice.png"
# ─────────────────────────────────────────────────────────────

if not HAS_C:
    print("ℹ  d13C column not found — skipping C/N plot.")
    print("   Add a d13C column to irms.csv to enable this section.")

elif not HAS_N:
    # --- Single-axis strip plot for δ¹³C alone ---
    print("ℹ  d15N not found — showing δ¹³C strip plot only.")
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.set_facecolor("#FAFAFA")

    # Reference spans
    ax.axvspan(-32, -22, alpha=0.15, color="#2E86AB",
               label="C3 oils (olives, sunflower, rape) −32 to −22‰")
    ax.axvspan(-16,  -9, alpha=0.15, color="#E76F51",
               label="C4 adulterants (corn oil) −16 to −9‰")

    # Sample points jittered on a single row
    y_jitter = np.random.default_rng(42).uniform(-0.1, 0.1, size=len(df))
    for i, (_, row) in enumerate(df.iterrows()):
        is_unk = "unknown" in str(row["SampleID"]).lower()
        col = "#6C3483" if is_unk else "#2C3E50"
        ax.scatter(row["d13C"], y_jitter[i], s=70, color=col,
                   edgecolors="white", linewidths=0.7, zorder=5)
        ax.annotate(str(row["SampleID"]),
                    xy=(row["d13C"], y_jitter[i]),
                    xytext=(row["d13C"], y_jitter[i] + 0.06),
                    fontsize=8, ha="center", color=col, fontweight="bold",
                    path_effects=[pe.withStroke(linewidth=2, foreground="white")],
                    zorder=6)

    ax.set_xlabel(r"$\delta^{13}$C (‰ vs VPDB)", fontsize=12)
    ax.set_yticks([])
    ax.set_title(r"$\delta^{13}$C Strip Plot — C3 vs C4 Oil Identification",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=8.5, loc="upper left", framealpha=0.9)
    ax.spines[["top", "right", "left"]].set_visible(False)
    plt.tight_layout()
    if SAVE_FIGURE_CN:
        plt.savefig(FIGURE_NAME_CN, dpi=FIGURE_DPI, bbox_inches="tight")
        print(f"\u2713 Saved {FIGURE_NAME_CN}")
    plt.show()

else:
    # --- Full 2D scatter: δ¹³C vs δ¹⁵N ---
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.set_facecolor("#FAFAFA")

    # Reference ellipses
    for region, vals in REF_CN.items():
        cx  = vals["d13C_mean"]
        cy  = vals["d15N_mean"]
        w   = 2 * vals["d13C_sd"]
        h   = 2 * vals["d15N_sd"]
        col = vals["colour"]

        ell = Ellipse((cx, cy), width=w, height=h,
                      facecolor=col, alpha=0.18, edgecolor=col,
                      linewidth=1.5, linestyle="--", zorder=1)
        ax.add_patch(ell)

        ax.text(cx, cy, region,
                fontsize=8.5, color=col, fontweight="bold",
                ha="center", va="center", zorder=3,
                path_effects=[pe.withStroke(linewidth=2, foreground="white")])

    # C3/C4 boundary line
    ax.axvline(-18, color="#888", lw=1.0, ls=":", alpha=0.7, zorder=2)
    ax.text(-18.2, ax.get_ylim()[0] + 0.5 if ax.get_ylim()[0] != 0 else 0.5,
            "C4/C3 boundary", fontsize=7.5, color="#888",
            ha="right", va="bottom", rotation=90)

    # Synthetic vs organic N guideline
    ax.axhline(0, color="#888", lw=0.8, ls=":", alpha=0.5, zorder=2)
    ax.text(ax.get_xlim()[0] + 0.3 if ax.get_xlim()[0] != 0 else -33,
            0.4, "Synthetic N fertiliser (δ¹⁵N ≈ 0‰)",
            fontsize=7.5, color="#888", va="bottom")

    # Sample points
    for _, row in df.iterrows():
        sid = str(row["SampleID"])
        x   = row["d13C"]
        y   = row["d15N"]
        is_unk = "unknown" in sid.lower() or "unk" in sid.lower()
        col    = "#6C3483" if is_unk else "#2C3E50"
        marker = "D"        if is_unk else "o"
        size   = 80         if is_unk else 60

        ax.scatter(x, y, marker=marker, s=size, color=col,
                   edgecolors="white", linewidths=0.8, zorder=5)
        ax.annotate(sid, xy=(x, y),
                    xytext=(x + 0.1, y + 0.3),
                    fontsize=8, color=col, fontweight="bold",
                    path_effects=[pe.withStroke(linewidth=2, foreground="white")],
                    zorder=6)

    ax.set_xlabel(r"$\delta^{13}$C (‰ vs VPDB)", fontsize=12)
    ax.set_ylabel(r"$\delta^{15}$N (‰ vs AIR)", fontsize=12)
    ax.set_title(r"$\delta^{13}$C vs $\delta^{15}$N — Agricultural Practice Profile",
                 fontsize=13, fontweight="bold", pad=10)

    legend_handles = [
        mpatches.Patch(facecolor=v["colour"], alpha=0.5, edgecolor=v["colour"],
                       linestyle="--", label=k)
        for k, v in REF_CN.items()
    ] + [
        plt.scatter([], [], marker="o", s=60, color="#2C3E50",
                    edgecolors="white", linewidths=0.8, label="Sample (labelled)"),
        plt.scatter([], [], marker="D", s=80, color="#6C3483",
                    edgecolors="white", linewidths=0.8, label="Sample (unknown)"),
    ]
    ax.legend(handles=legend_handles, title="Reference regions (±1 SD)",
              title_fontsize=8.5, fontsize=8.5, loc="upper right", framealpha=0.9)

    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, linestyle=":", alpha=0.4)

    ax.text(0.5, -0.10,
            "Ellipses: ±1 SD reference regions (literature). "
            "C4 adulterants (corn oil) appear far right (δ¹³C > −18‰).",
            transform=ax.transAxes, ha="center", fontsize=8, color="grey")

    plt.tight_layout()
    if SAVE_FIGURE_CN:
        plt.savefig(FIGURE_NAME_CN, dpi=FIGURE_DPI, bbox_inches="tight")
        print(f"\u2713 Saved {FIGURE_NAME_CN}")
    plt.show()

### How to interpret the δ¹³C / δ¹⁵N plot

**C3 vs C4 discrimination:**  
Olive oil is a C3 plant product (δ¹³C typically −32 to −22‰). Corn oil, a common adulterant,  
is a C4 product (δ¹³C typically −16 to −9‰). A value more positive than −18‰ is a strong  
indicator of C4 adulteration.

**Important limitation:**  
Sunflower and rapeseed oils are also C3 plants with similar δ¹³C ranges to olive oil —  
δ¹³C alone cannot detect these adulterants. Additional authentication tools (FTIR, NMR,  
fatty acid profiling) are needed for complete quality assurance.

**Nitrogen as a practice indicator:**  
Organic certification claims can be supported (but not proven conclusively) by elevated δ¹⁵N  
(>+5‰), consistent with manure-based fertilisation. Synthetic fertiliser gives δ¹⁵N ≈ 0‰.

## 7 · Origin Assignment

For each sample, compute its Euclidean distance in the δ²H / δ¹⁸O space  
to each reference region centre (normalised by the regional SD).  
The nearest region is the **most probable origin** — displayed as a table.

This is a simplified nearest-centroid classifier.  
A full forensic authentication would use validated LDA or QDA models  
trained on multi-year reference databases.

In [ ]:
# ============================================================
# NEAREST-CENTROID ORIGIN ASSIGNMENT
# ============================================================
# Mahalanobis-like distance using per-axis SD scaling:
#   d = sqrt( ((d18O_sample - d18O_ref) / SD_d18O)^2
#           + ((d2H_sample  - d2H_ref)  / SD_d2H )^2 )
#
# A lower score means the sample is closer to that region.
# A score < 1 means the sample falls within ±1 SD on both axes.

print("Origin assignment (nearest-centroid, normalised Euclidean distance)\n")
print(f"  {'SampleID':<18}  {'d18O':>6}  {'d2H':>7}  {'Nearest region':<14}  "
      f"{'Score':>6}  {'2nd closest':<14}  {'Score':>6}")
print("  " + "─" * 82)

assignments = []

for _, row in df.iterrows():
    sid   = str(row["SampleID"])
    x18   = row["d18O"]
    x2H   = row["d2H"]

    # Compute normalised distance to each reference region
    distances = {}
    for region, vals in REF_HO.items():
        dx = (x18 - vals["d18O_mean"]) / vals["d18O_sd"]
        dy = (x2H - vals["d2H_mean"])  / vals["d2H_sd"]
        distances[region] = np.sqrt(dx**2 + dy**2)

    sorted_regions = sorted(distances, key=distances.get)
    best   = sorted_regions[0]
    second = sorted_regions[1] if len(sorted_regions) > 1 else "—"

    assignments.append({
        "SampleID":       sid,
        "d18O":           x18,
        "d2H":            x2H,
        "Assigned origin": best,
        "Score":          round(distances[best], 2),
        "2nd closest":    second,
        "Score (2nd)":    round(distances[second], 2) if second != "—" else None,
    })

    # Flag ambiguous assignments (best and second within 0.3 distance units)
    ambiguous = (
        len(sorted_regions) > 1
        and abs(distances[best] - distances[second]) < 0.3
    )
    flag = " ⚠ ambiguous" if ambiguous else ""

    print(f"  {sid:<18}  {x18:6.2f}  {x2H:7.1f}  {best:<14}  "
          f"{distances[best]:6.2f}  {second:<14}  "
          f"{distances[second] if second != '—' else '—':>6}{flag}")

results_df = pd.DataFrame(assignments)
print("\n\u2713 Assignment complete. See table above.")
print("\n  Note: Score < 1.0 = sample within ±1 SD of regional centroid (good match)")
print("        Score > 2.0 = sample outside ±2 SD (poor match — consider mislabelling)")

## 8 · Discussion Questions

Use the plots and assignment table above to work through the following questions  
in your practical report. Refer to the lecture slides and Torres-Cobos et al. (2025)  
as supporting references.

---

**Q1.** How can stable isotope ratios be used to identify the geographic origin of an olive oil  
sample? Explain the physical and chemical processes that create regional differences in  
δ¹⁸O and δ²H.

**Q2.** What are the strengths and weaknesses of IRMS for geographic authentication?  
What analytical techniques could be used alongside IRMS to strengthen a fraud case?

**Q3.** Based on the δ²H / δ¹⁸O plot and the assignment table, what is the most probable  
origin of each unknown sample? How confident are you in this assignment, and why?

**Q4.** Why might irrigation practices affect the reliability of δ¹⁸O and δ²H for  
geographic authentication? How could a reference database be designed to account  
for this?

**Q5.** The 2018/19 Italian harvest showed higher misclassification rates than other years.  
What meteorological conditions might explain this, and what does it imply for  
building operationally robust authentication systems?

---

> **Key reference:**  
> Torres-Cobos et al. (2025). *Food Chemistry*, doi:10.1016/j.foodchem.2024.141998  
> Study of ~400 virgin olive oil samples from six Mediterranean countries across  
> four harvest seasons (2016–2020).